# 讽刺/非讽刺 分类

In [13]:
import json
import os
import shutil
from tqdm import tqdm  # 如果没有安装 tqdm，可以去掉这行，或者运行 pip install tqdm

# ================= 配置区域 =================
# 1. 你的 JSON 文件路径
json_file_path = 'sarcasm_data.json'

# 2. 存放原始视频的文件夹路径
source_video_dir = 'utterances_final' 

# 3. 你希望保存分类结果的目标文件夹
output_base_dir = '/projects/kzh/VIBE_gaze/Sarcasm/utterances_L2CS_TF' 

# 4. 视频文件的后缀名 (MUStARD 数据集通常是 .mp4)
file_extension = '.mp4'
# ===========================================

def sort_videos():
    # 1. 检查并创建目标文件夹 (dataset_sorted/true 和 dataset_sorted/false)
    true_dir = os.path.join(output_base_dir, 'true')
    false_dir = os.path.join(output_base_dir, 'false')
    
    os.makedirs(true_dir, exist_ok=True)
    os.makedirs(false_dir, exist_ok=True)
    
    print(f"创建目标文件夹:\n - {true_dir}\n - {false_dir}")

    # 2. 读取 JSON 数据
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"错误: 找不到 JSON 文件: {json_file_path}")
        return

    print(f"成功加载 JSON，共包含 {len(data)} 条数据。开始处理...")

    # 3. 遍历数据并复制文件
    # 如果安装了 tqdm，把下面的 range 改为 tqdm(data.items()) 可以显示进度条
    # for task_id, info in tqdm(data.items(), desc="Processing"):
    
    count_success = 0
    count_missing = 0

    for task_id, info in data.items():
        # 获取标签
        is_sarcastic = info.get('sarcasm')
        
        # 构造源文件路径 (例如: 1_70.mp4)
        video_filename = task_id + file_extension
        src_path = os.path.join(source_video_dir, video_filename)
        
        # 确定目标路径
        if is_sarcastic:
            dest_folder = true_dir
        else:
            dest_folder = false_dir
            
        dest_path = os.path.join(dest_folder, video_filename)
        
        # 检查源文件是否存在
        if os.path.exists(src_path):
            # 复制文件 (copy2 会保留文件的元数据，如创建时间)
            shutil.copy2(src_path, dest_path)
            count_success += 1
        else:
            print(f"[警告] 文件缺失: {video_filename} (Label: {is_sarcastic})")
            count_missing += 1

    print("-" * 30)
    print("处理完成！")
    print(f"成功分类: {count_success} 个视频")
    print(f"文件缺失: {count_missing} 个视频")
    print(f"文件保存在: {output_base_dir}")

if __name__ == "__main__":
    sort_videos()

创建目标文件夹:
 - /projects/kzh/VIBE_gaze/Sarcasm/utterances_L2CS_TF/true
 - /projects/kzh/VIBE_gaze/Sarcasm/utterances_L2CS_TF/false
成功加载 JSON，共包含 690 条数据。开始处理...
------------------------------
处理完成！
成功分类: 690 个视频
文件缺失: 0 个视频
文件保存在: /projects/kzh/VIBE_gaze/Sarcasm/utterances_L2CS_TF


# head orientation

In [11]:
import cv2
import csv
import os
import math
from sixdrepnet import SixDRepNet
from tqdm import tqdm

# ================= 配置区域 =================
# 输入视频路径 (建议使用你之前超分增强过的视频)
video_path = 'utterances_final/2_97.mp4'

# 输出视频路径
output_video_path = '/projects/kzh/VIBE_gaze/Sarcasm/utterances_head/2_97_pose.mp4'

# 输出数据路径 (CSV文件，保存具体的角度数值)
output_csv_path = '/projects/kzh/VIBE_gaze/Sarcasm/utterances_head/2_97_data.csv'

# 确保输出文件夹存在
os.makedirs(os.path.dirname(output_video_path), exist_ok=True)

project_root = '/projects/kzh/VIBE_gaze/Sarcasm'
weight_path = os.path.join(project_root, 'checkpoints', '6DRepNet_300W_LP_AFLW2000.pth')
# ===========================================

def process_video_pose():
    # 1. 初始化模型 (GPU会自动使用)
    # SixDRepNet 会自动下载权重
    model = SixDRepNet(dict_path=weight_path)
    
    # 2. 打开视频
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Cannot open video {video_path}")
        return

    # 获取视频属性
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # 3. 设置视频写入器
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    # 4. 准备 CSV 文件写入
    csv_file = open(output_csv_path, 'w', newline='')
    csv_writer = csv.writer(csv_file)
    # 写入表头: 帧号, Pitch(抬头/低头), Yaw(左右转), Roll(歪头)
    csv_writer.writerow(["frame", "pitch", "yaw", "roll"])

    print(f"开始处理视频: {video_path}")
    print(f"总帧数: {total_frames}")

    # 5. 逐帧处理
    frame_idx = 0
    pbar = tqdm(total=total_frames)

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_idx += 1
        
        try:
            # SixDRepNet 的 predict 方法：
            # 1. 内部会自动调用人脸检测器检测人脸
            # 2. 返回 Pitch, Yaw, Roll
            # 注意：如果画面有多个人脸，默认库通常返回置信度最高的一个，或者你需要修改源码支持多脸。
            # 这里假设主要针对说话者 (Speaker)。
            pitch, yaw, roll = model.predict(frame)

            # --- 保存数据 ---
            # 注意：不同库返回的数据格式可能略有不同(Tensor或Float)
            # 通常 SixDRepNet 返回的是 numpy scalar 或 float
            csv_writer.writerow([frame_idx, float(pitch), float(yaw), float(roll)])

            # --- 可视化绘制 ---
            # draw_axis(img, yaw, pitch, roll, tdx=None, tdy=None, size=100)
            # 它会自动找到人脸中心并画轴
            model.draw_axis(frame, yaw, pitch, roll)
            
            # (可选) 在画面上打印数值
            text = f"P:{pitch:.1f} Y:{yaw:.1f} R:{roll:.1f}"
            cv2.putText(frame, text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 
                        1, (0, 255, 0), 2)

        except Exception as e:
            # 如果这一帧没检测到人脸，model.predict 可能会报错
            # 或者返回 None，具体取决于库的版本。
            # 这种情况下，我们通常填入空值或上一帧的值，或者直接跳过绘制
            # print(f"Frame {frame_idx}: No face detected or error: {e}")
            csv_writer.writerow([frame_idx, "NaN", "NaN", "NaN"])
            pass

        # 写入视频帧
        out.write(frame)
        pbar.update(1)

    # 6. 清理资源
    cap.release()
    out.release()
    csv_file.close()
    pbar.close()
    print("\n处理完成！")
    print(f"视频已保存: {output_video_path}")
    print(f"数据已保存: {output_csv_path}")

if __name__ == "__main__":
    process_video_pose()

开始处理视频: utterances_final/2_97.mp4
总帧数: 41


  0%|          | 0/41 [00:00<?, ?it/s]/tmp/ipykernel_1224643/723991640.py:77: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  csv_writer.writerow([frame_idx, float(pitch), float(yaw), float(roll)])
100%|██████████| 41/41 [00:00<00:00, 53.20it/s]


处理完成！
视频已保存: /projects/kzh/VIBE_gaze/Sarcasm/utterances_head/2_97_pose.mp4
数据已保存: /projects/kzh/VIBE_gaze/Sarcasm/utterances_head/2_97_data.csv


# 单次推理测试

In [16]:
import cv2
import ollama
from PIL import Image
import io
import os
import glob

# ================= 配置区域 =================
MODEL_NAME = "qwen2.5vl:7b"  

# VIDEO_PATH = "utterances_gazelle/1_276.mp4"
# VIDEO_PATH = 'utterances_L2CS_smoothed/1_276.mp4'
# VIDEO_PATH = 'utterances_final/1_276.mp4'

# VIDEO_PATH = 'utterances_eyeroll/1_276.mp4'
VIDEO_PATH = 'mediapipe_output.mp4'

KEYFRAME_ROOT = "keyframe_smooth/"

# MAX_IMAGE_WIDTH = 920
JPEG_QUALITY = 100

# 如果视频长 10 秒，SAMPLES=10 代表每秒截一张
TARGET_FRAME_COUNT = 20
# ===========================================

def load_keyframes_from_dir(task_id):
    """
    从关键帧目录加载已保存的关键帧图片，并转为字节流
    """
    # 拼接路径，例如 ./keyframes_output/1_60
    keyframe_dir = os.path.join(KEYFRAME_ROOT, task_id)
    
    if not os.path.exists(keyframe_dir):
        # 尝试容错：有些系统可能把 ID 中的 _ 替换成了其他字符，或者文件夹结构不同
        # 这里仅作示例，如果路径严格匹配则不需要
        print(f"⚠️ 关键帧目录不存在: {keyframe_dir}")
        return []
    
    # 获取所有 jpg 文件并按名称排序 (确保时间顺序)
    # 假设文件名是按顺序命名的，如 001.jpg, 002.jpg
    jpg_files = sorted(glob.glob(os.path.join(keyframe_dir, "*.jpg")))
    
    if not jpg_files:
        print(f"⚠️ 在 {keyframe_dir} 中未找到 JPG 文件")
        return []
    
    frame_images = []
    
    for jpg_path in jpg_files:
        try:
            with Image.open(jpg_path) as pil_img:
                # 1. 缩放图片以节省显存 (保持宽高比)
                # 如果图片已经很小，可以跳过这一步
                # if pil_img.width > MAX_IMAGE_WIDTH:
                #     ratio = MAX_IMAGE_WIDTH / float(pil_img.width)
                #     new_height = int(float(pil_img.height) * ratio)
                #     pil_img = pil_img.resize((MAX_IMAGE_WIDTH, new_height), Image.Resampling.LANCZOS)
                
                # 2. 转为 JPEG 字节流
                img_byte_arr = io.BytesIO()
                pil_img.save(img_byte_arr, format='JPEG', quality=JPEG_QUALITY)
                frame_data = img_byte_arr.getvalue()
                frame_images.append(frame_data)
        except Exception as e:
            print(f"❌ 读取文件失败 {jpg_path}: {e}")
            continue
    
    return frame_images


def extract_frames(video_path, target_count=TARGET_FRAME_COUNT):
    """
    从视频中均匀抽取指定数量的帧，并转换为字节流
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ 无法打开视频: {video_path}")
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    duration = total_frames / fps if fps > 0 else 0
    
    print(f"🎬 视频信息: 时长 {duration:.2f}s | 总帧数 {total_frames} | FPS {fps}")
    
    # 计算抽帧间隔
    step = max(1, total_frames // target_count)
    
    frame_images = [] 
    
    count = 0
    saved_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        # 按照步长抽帧
        if count % step == 0 and saved_count < target_count:
            # OpenCV 默认是 BGR，需要转为 RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            
            # 为了加快速度和节省显存，可以将图片缩小，例如最大边长 720 或 512
            # pil_img.thumbnail((960, 540)) 
            
            # 转为字节流
            img_byte_arr = io.BytesIO()
            pil_img.save(img_byte_arr, format='JPEG', quality=JPEG_QUALITY)
            image_bytes = img_byte_arr.getvalue()
            
            frame_images.append(image_bytes)
            saved_count += 1
            
        count += 1

    cap.release()
    print(f"抽取 {len(frame_images)} 帧用于分析")
    return frame_images


def analyze_video_gaze(video_path):
    frames = extract_frames(video_path, target_count=TARGET_FRAME_COUNT)
    # frames = load_keyframes_from_dir(os.path.splitext(os.path.basename(video_path))[0])
    
    if not frames:
        return

    # prompt = (
    #     f"""
    #     You are a Human Gaze and Analysis Expert.
    #     Your goal is to extract the main speaker's gaze event sequences from the videos and convert into a single, continuous, human-centric sentence that highlights the key gaze patterns for downstream Sarcasm recognition tasks. 
    #     The line connects the eye and object being looked at, indicating gaze direction and behavior.
    #     # Input semantics (from symbolic gaze reference)
    #     - Event types: Fixation, Saccade, SmoothPursuit
    #     - Strictly use the exact tokens when referring to event types: "fixation", "saccade", "smooth pursuit" (avoid synonyms like glance, gaze shift, tracking)
    #     - Use qualitative words implied by labels only: duration → brief/short/moderate/long; amplitude → small/medium/large; speed → slow/fast/very fast (pursuit may use steady)

    #     # Style & constraints
    #     - Summarize the most salient gaze patterns of the main speaker [LEONARD];
    #     - Maintain temporal continuity across gaze shifts;
    #     - Only describe gaze behaviors objectively in sequence and less than 3 sentences;
    #     """
    # )

    # prompt = (
    #     f"""
        # You are a Gaze Tracking Assistant.
        # Your task is to describe the gaze behavior of [LEONARD] in [BBT] in the whole video based on the green line.
        
        # # VISUAL INSTRUCTIONS:
        # 1. **Start Point of the green line**: The person in the green bounding box.
        # 2. **End Point of the green line**: The object/person beeing seen.
        # 3. If the green lines disappear, it means the speaker's gaze "saccade" or "fixation" at something out of view.

        # # VOCABULARY RULES:
        # - Use ONLY these terms for events: "fixation" (steady gaze on a target), "saccade" (quick shift between targets), "smooth pursuit" (following a moving target).
        # - Use implied adjectives: brief/long (duration), small/large (amplitude).

        # # OUTPUT FORMAT:
        # - Provide a chronological sentence less than 3 sentences.
    
    #     """
    # )

    # prompt = (
    #     """
    #     You are a video-based Gaze Analysis Expert.
    #     Task: Determine whether Leonard shows (1) a clear eye-roll or (2) a contemptuous squint / side-eye at any time in the video.
    #     Procedure:
    #     1.	For each moment, describe objective cues only (gaze direction, eyelid aperture, head yaw/pitch, duration).
    #     2.	Decide: Eye-roll present? Squint/side-eye present?
    #     Output format (no extra text):
    #     eye_roll: Yes/No
    #     side_eye_squint: Yes/No
    #     final: Yes if either is Yes, else No
    #     """
    # )

    target_behavior = "eye-roll"  # 或者是 "eye-roll"

    prompt = (f"""
    You are a Human Behavior Analysis Expert specialized in detecting subtle non-verbal cues.
    I have detected a potential '{target_behavior}' signal in the red bounding box. 
    Your task is to VERIFY if this detection is a genuine social signal or a false positive (e.g., normal looking around, blinking).

    Please analyze the video focusing on the red bounding box:
    1. **Head vs. Eye Direction**: Does the person's head rotate with the eyes? (For side-eye, the head stays still while eyes move to the corner. For eye-roll, the eyes move upward/circularly independent of head motion).
    (note that the eye-roll is usually mixed with constant downward gaze or closing of the eyes)
    2. **Sclera Visibility**: Can you see a significant amount of the white part (sclera) of the eye?

    Based on the above, answer with the following format:
    - Observation: [Describe the eye and head movement briefly]
    - Verdict: [True Positive / False Positive]
    """)


    print("推理中")
    
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[{
                'role': 'user',
                'content': prompt,
                'images': frames  # 这里传入所有帧
            }],
                options={
                    'temperature': 0.1,
                    'num_ctx': 30000
                }
        )

        gaze_desc = response['message']['content']
        
        # print("\n" + "="*20 + " 识别结果 " + "="*20)
        # print(response['message']['content'])
        # print("="*50)

        return response['message']['content']
        
    except Exception as e:
        print(f"❌ 推理出错: {e}")


def analyze_video(gaze_desc,video_path=VIDEO_PATH):
    frames = extract_frames(video_path, target_count=TARGET_FRAME_COUNT)

    prompt = (
        f"""
        You are an expert in analyzing social interactions, specifically detecting sarcasm.
        # Target utterance: [It's just a privilege to watch your mind at work.]
        # Gaze description: {gaze_desc}
        # Speaker: [SHELDON]

        # Task:
        Analyze the speaker's gaze description, utterance.
        Determine if the target utterance is SARCASTIC or NOT SARCASTIC.

        ### Response Format:
        1. Thinking out loud: Analyze the gaze description, and the incongruity between gaze and utterance (less than 3 sentences).
        2. Final Answer: Return strictly valid JSON format: {{\"sarcasm\": true}} or {{\"sarcasm\": false}}.
        """
    )

    print("推理中")
    
    try:
        response = ollama.chat(
            model="qwen2.5vl:7b",
            messages=[{
                'role': 'user',
                'content': prompt,
                'images': frames  # 这里传入帧
            }],
                options={
                    'temperature': 0.1,
                    # 'num_ctx': 30000
                }
        )
        
        # print("\n" + "="*20 + " 识别结果 " + "="*20)
        # print(response['message']['content'])
        # print("="*50)

        return response['message']['content']
        
    except Exception as e:
        print(f"❌ 推理出错: {e}")


task_id = "1_276"
# gaze_desc = analyze_video_gaze(task_id)

gaze_desc = analyze_video_gaze(VIDEO_PATH)
print(gaze_desc)

# gaze_desc = "SHELDON maintains a long fixation, but a saccade occurs when saying."

# analyze_video(gaze_desc)

🎬 视频信息: 时长 5.78s | 总帧数 133 | FPS 23
抽取 20 帧用于分析
推理中
    - Observation: The person's eyes are moving upward and slightly to the right, while the head remains relatively still. The eyes are not fully closed, and the white part (sclera) of the eye is visible.
    - Verdict: False Positive

    The upward eye movement and slight head rotation are more indicative of a side-eye glance rather than an eye-roll. The visibility of the sclera and the upward gaze suggest that the person is looking at something or someone to the side, rather than rolling their eyes.


In [2]:
import json
import os

# ================= 配置 =================
# INPUT_FILE = "./results/glm_gaze_qwen_inference_wrong.jsonl"  
INPUT_FILE = "results/cv_gaze_qwenvl_inference.jsonl"
# ERROR_FILE = "./results/error_cases_eyeroll_01.json"          # 错误样本输出路径
# =======================================

def evaluate_and_extract_errors():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ 文件不存在: {INPUT_FILE}")
        return

    # 初始化计数器
    stats = {
        "total": 0,
        "correct": 0,
        "tp": 0, # 真阳性 (GT=True, Pred=True)
        "tn": 0, # 真阴性 (GT=False, Pred=False)
        "fp": 0, # 假阳性 (GT=False, Pred=True) -> 误报
        "fn": 0  # 假阴性 (GT=True, Pred=False) -> 漏报
    }
    
    error_cases = []

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            
            item = json.loads(line)
            gt = item.get("ground_truth")
            pred = item.get("sarcasm_prediction")

            # 跳过没有标签的数据
            if gt is None: continue

            stats["total"] += 1

            if gt == pred:
                stats["correct"] += 1
                if gt is True: stats["tp"] += 1
                else: stats["tn"] += 1
            else:
                # 记录错误类型
                if pred is True: 
                    stats["fp"] += 1
                    item["error_type"] = "False Positive (误报)"
                else: 
                    stats["fn"] += 1
                    item["error_type"] = "False Negative (漏报)"
                
                # 加入错误列表
                error_cases.append(item)

    # --- 计算指标 ---
    if stats["total"] == 0:
        print("数据为空")
        return

    acc = stats["correct"] / stats["total"]
    # 防止除以零
    precision = stats["tp"] / (stats["tp"] + stats["fp"]) if (stats["tp"] + stats["fp"]) > 0 else 0
    recall = stats["tp"] / (stats["tp"] + stats["fn"]) if (stats["tp"] + stats["fn"]) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # --- 打印报告 ---
    print(f"\n{'='*30}")
    print(f"📊 评估结果 (样本数: {stats['total']})")
    print(f"{'='*30}")
    print(f"✅ Accuracy (准确率):  {acc:.2%}")
    print(f"🎯 Precision (查准率): {precision:.2%}")
    print(f"🔎 Recall    (查全率): {recall:.2%}")
    print(f"⚖️  F1-Score:         {f1:.2%}")
    print(f"{'-'*30}")
    print(f"详细统计:")
    print(f"  TP (讽刺且预测对): {stats['tp']}")
    print(f"  TN (正常且预测对): {stats['tn']}")
    print(f"  FP (误判为讽刺):   {stats['fp']} (模型太敏感)")
    print(f"  FN (漏判讽刺):     {stats['fn']} (模型没识别出)")
    print(f"{'='*30}")

    # --- 保存错误文件 ---
    # 保存为标准 JSON 列表格式，带缩进，方便人眼查看
    # with open(ERROR_FILE, 'w', encoding='utf-8') as f:
    #     json.dump(error_cases, f, indent=4, ensure_ascii=False)
    
    # print(f"\n📝 已将 {len(error_cases)} 个错误案例保存至: {ERROR_FILE}")

if __name__ == "__main__":
    evaluate_and_extract_errors()


📊 评估结果 (样本数: 183)
✅ Accuracy (准确率):  69.40%
🎯 Precision (查准率): 65.91%
🔎 Recall    (查全率): 88.78%
⚖️  F1-Score:         75.65%
------------------------------
详细统计:
  TP (讽刺且预测对): 87
  TN (正常且预测对): 40
  FP (误判为讽刺):   45 (模型太敏感)
  FN (漏判讽刺):     11 (模型没识别出)
